# Importance Sampling: An Illustrative Example

Importance sampling is the key tool behind **off-policy** methods. We want an expectation under one distribution $p_0$ (the *target*), but our samples come from a different distribution $p_1$ (the *behavior*).

Because

$$
\mathbb{E}_{X\sim p_0}[X] = \sum_{x} p_0(x)\,x = \sum_{x} p_1(x)\underbrace{\frac{p_0(x)}{p_1(x)}\,x}_{f(x)} = \mathbb{E}_{X\sim p_1}[f(X)], \qquad (10.9)
$$

we can estimate it from samples $\{x_i\}_{i=1}^n$ drawn from $p_1$:

$$
\mathbb{E}_{X\sim p_0}[X] \approx \bar f = \frac{1}{n}\sum_{i=1}^n \underbrace{\frac{p_0(x_i)}{p_1(x_i)}}_{\text{importance weight}} x_i. \qquad (10.10)
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Two distributions over $X \in \{+1, -1\}$

| | $p(X=+1)$ | $p(X=-1)$ | $\mathbb{E}[X]$ |
|---|---|---|---|
| target $p_0$ | 0.5 | 0.5 | $0$ |
| behavior $p_1$ | 0.8 | 0.2 | $0.6$ |

Our goal is to estimate $\mathbb{E}_{X\sim p_0}[X] = 0$ using **only samples drawn from $p_1$**.

In [ ]:
values = np.array([+1, -1])
p0 = np.array([0.5, 0.5])   # target distribution
p1 = np.array([0.8, 0.2])   # behavior distribution (generates the samples)

N_SAMPLES = 200
SEED = 86

print(f"E_p0[X] = {np.sum(p0 * values):.2f}")
print(f"E_p1[X] = {np.sum(p1 * values):.2f}")

## 2. Sample from $p_1$ and compute both estimates

After each new sample we compute two running estimates:

- the **plain average** $\bar x_n = \frac{1}{n}\sum_{i=1}^n x_i$, which converges to $\mathbb{E}_{X\sim p_1}[X] = 0.6$;
- the **importance-sampling average** $\bar f_n = \frac{1}{n}\sum_{i=1}^n \frac{p_0(x_i)}{p_1(x_i)}\, x_i$, which converges to $\mathbb{E}_{X\sim p_0}[X] = 0$.

In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(values), size=N_SAMPLES, p=p1)   # index 0 -> +1, index 1 -> -1
x = values[idx]

weights = p0[idx] / p1[idx]    # importance weights p0(x_i) / p1(x_i)

n = np.arange(1, N_SAMPLES + 1)
plain_avg = np.cumsum(x) / n
is_avg = np.cumsum(weights * x) / n

print(f"Fraction of +1 samples:                  {np.mean(x == 1):.3f}")
print(f"Plain average after {N_SAMPLES} samples:       {plain_avg[-1]:.3f}   (-> 0.6)")
print(f"Importance sampling after {N_SAMPLES} samples: {is_avg[-1]:.3f}   (-> 0)")

## 3. Plot

The dots are the samples. The dotted line is the plain average, and the solid line is the importance-sampling estimate.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(n, x, "o", markersize=3, alpha=0.6, label="samples")
plt.plot(n, plain_avg, ":", linewidth=2, label="average")
plt.plot(n, is_avg, "-", linewidth=2, label="importance sampling")
plt.axhline(0.6, color="gray", linewidth=0.8, alpha=0.5)
plt.axhline(0.0, color="gray", linewidth=0.8, alpha=0.5)
plt.ylim(-1.5, 1.5)
plt.xlabel("Sample index")
plt.title(r"Estimating $E_{p_0}[X]$ with samples drawn from $p_1$")
plt.legend(loc="lower center", ncol=3)
plt.grid(alpha=0.3)
plt.show()

### Why does the weighted average work?

| $x$ | $p_0(x)$ | $p_1(x)$ | weight $p_0(x)/p_1(x)$ |
|---|---|---|---|
| $+1$ | 0.5 | 0.8 | 0.625 |
| $-1$ | 0.5 | 0.2 | 2.5 |

$p_1$ draws $+1$ more often than $p_0$ would, so each $+1$ is **down-weighted**. $-1$ is rare under $p_1$, so each $-1$ counts **2.5 times**. In expectation the two effects cancel exactly:

$$
0.8\cdot(0.625)(+1) + 0.2\cdot(2.5)(-1) = 0.5 - 0.5 = 0 .
$$